In [3]:
!pip install plotly

In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from IPython.display import display, HTML
from google.colab import files

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("Dataset imported successfully!")

Saving task_3 data validation.zip to task_3 data validation.zip
Dataset imported successfully!


In [9]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe(include="all"))

Dataset Shape: (9537, 11)

Column Names:
['session_id', 'network_packet_size', 'protocol_type', 'login_attempts', 'session_duration', 'encryption_used', 'ip_reputation_score', 'failed_logins', 'browser_type', 'unusual_time_access', 'attack_detected']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9537 entries, 0 to 9536
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   session_id           9537 non-null   object 
 1   network_packet_size  9537 non-null   int64  
 2   protocol_type        9537 non-null   object 
 3   login_attempts       9537 non-null   int64  
 4   session_duration     9537 non-null   float64
 5   encryption_used      7571 non-null   object 
 6   ip_reputation_score  9537 non-null   float64
 7   failed_logins        9537 non-null   int64  
 8   browser_type         9537 non-null   object 
 9   unusual_time_access  9537 non-null   int64  
 10  attack_detected

,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
count,9537,9537.000000,9537,9537.000000,9537.000000,7571,9537.000000,9537.000000,9537,9537.000000,9537.000000
unique,9537,NaN,3,NaN,NaN,2,NaN,NaN,5,NaN,NaN
top,SID_00001,NaN,TCP,NaN,NaN,AES,NaN,NaN,Chrome,NaN,NaN
freq,1,NaN,6624,NaN,NaN,4706,NaN,NaN,5137,NaN,NaN
mean,NaN,500.430639,NaN,4.032086,792.745312,NaN,0.331338,1.517773,NaN,0.149942,0.447101
std,NaN,198.379364,NaN,1.963012,786.560144,NaN,0.177175,1.033988,NaN,0.357034,0.497220
min,NaN,64.000000,NaN,1.000000,0.500000,NaN,0.002497,0.000000,NaN,0.000000,0.000000
25%,NaN,365.000000,NaN,3.000000,231.953006,NaN,0.191946,1.000000,NaN,0.000000,0.000000
50%,NaN,499.000000,NaN,4.000000,556.277457,NaN,0.314778,1.000000,NaN,0.000000,0.000000
75%,NaN,635.000000,NaN,5.000000,1105.380602,NaN,0.453388,2.000000,NaN,0.000000,1.000000


In [10]:
print("Missing Values Before Cleaning:")
print(df.isnull().sum())

df["encryption_used"] = (
    df["encryption_used"].astype("object").fillna("Unknown")
)

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

Missing Values Before Cleaning:
session_id                0
network_packet_size       0
protocol_type             0
login_attempts            0
session_duration          0
encryption_used        1966
ip_reputation_score       0
failed_logins             0
browser_type              0
unusual_time_access       0
attack_detected           0
dtype: int64

Missing Values After Cleaning:
session_id             0
network_packet_size    0
protocol_type          0
login_attempts         0
session_duration       0
encryption_used        0
ip_reputation_score    0
failed_logins          0
browser_type           0
unusual_time_access    0
attack_detected        0
dtype: int64


In [11]:
df["attack_detected"] = pd.to_numeric(
    df["attack_detected"], errors="coerce"
)

df["attack_label"] = df["attack_detected"].map({
    0: "No Attack",
    1: "Attack Detected"
})

print(df["attack_label"].value_counts())

attack_label
No Attack          5273
Attack Detected    4264
Name: count, dtype: int64


In [12]:
total_sessions = len(df)

attacks = (df["attack_detected"] == 1).sum()

normal_sessions = (df["attack_detected"] == 0).sum()

attack_rate = (attacks / total_sessions) * 100

print("Total Sessions:", total_sessions)
print("Attacks Detected:", attacks)
print("Normal Sessions:", normal_sessions)
print("Attack Rate:", round(attack_rate, 2), "%")

Total Sessions: 9537
Attacks Detected: 4264
Normal Sessions: 5273
Attack Rate: 44.71 %


In [13]:
kpi_html = f"""
<div style="display:flex; gap:15px; flex-wrap:wrap;">

<div style="background:#1f2937;color:white;padding:20px;border-radius:10px;flex:1">
<h3>Total Sessions</h3>
<h1>{total_sessions:,}</h1>
</div>

<div style="background:#991b1b;color:white;padding:20px;border-radius:10px;flex:1">
<h3>Attacks Detected</h3>
<h1>{attacks:,}</h1>
</div>

<div style="background:#166534;color:white;padding:20px;border-radius:10px;flex:1">
<h3>Normal Sessions</h3>
<h1>{normal_sessions:,}</h1>
</div>

<div style="background:#1d4ed8;color:white;padding:20px;border-radius:10px;flex:1">
<h3>Attack Rate</h3>
<h1>{attack_rate:.2f}%</h1>
</div>

</div>
"""

display(HTML(kpi_html))

In [14]:
fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=(
        "Attack Distribution",
        "Protocol-wise Attack Analysis",
        "Login Attempts by Attack Status",
        "Session Duration Distribution",
        "Encryption Usage",
        "IP Reputation Score Analysis"
    ),
    specs=[
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}]
    ],
    vertical_spacing=0.12
)

In [15]:
attack_counts = df["attack_label"].value_counts()

fig.add_trace(
    go.Pie(
        labels=attack_counts.index,
        values=attack_counts.values,
        hole=0.4
    ),
    row=1,
    col=1
)

In [16]:
protocol_data = pd.crosstab(
    df["protocol_type"],
    df["attack_label"]
)

for column in protocol_data.columns:

    fig.add_trace(
        go.Bar(
            x=protocol_data.index.astype(str),
            y=protocol_data[column],
            name=column
        ),
        row=1,
        col=2
    )

In [17]:
for label in df["attack_label"].dropna().unique():

    subset = df[df["attack_label"] == label]

    fig.add_trace(
        go.Box(
            y=subset["login_attempts"],
            name=label
        ),
        row=2,
        col=1
    )

In [18]:
for label in df["attack_label"].dropna().unique():

    subset = df[df["attack_label"] == label]

    fig.add_trace(
        go.Histogram(
            x=subset["session_duration"],
            name=label,
            opacity=0.6
        ),
        row=2,
        col=2
    )

In [19]:
encryption_data = pd.crosstab(
    df["encryption_used"],
    df["attack_label"]
)

for column in encryption_data.columns:

    fig.add_trace(
        go.Bar(
            x=encryption_data.index.astype(str),
            y=encryption_data[column],
            name=column,
            showlegend=False
        ),
        row=3,
        col=1
    )

In [20]:
for label in df["attack_label"].dropna().unique():

    subset = df[df["attack_label"] == label]

    fig.add_trace(
        go.Box(
            y=subset["ip_reputation_score"],
            name=label,
            showlegend=False
        ),
        row=3,
        col=2
    )

In [21]:
fig.update_layout(
    title={
        "text": "CYBERSECURITY & CYBER ATTACK ANALYSIS DASHBOARD",
        "x": 0.5
    },
    height=1200,
    width=1100,
    template="plotly_dark",
    barmode="group"
)

fig.show()

In [22]:
fig.write_html("Cybersecurity_Dashboard.html")

files.download("Cybersecurity_Dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>